# DAT to CSV Converter
This notebook combines .dat files that start with the same 3 digits into CSV format.

In [6]:
import os
import pandas as pd
import glob
from pathlib import Path
from collections import defaultdict

In [7]:
def read_dat_file(file_path):
    """
    Read a .dat file and return a pandas DataFrame
    Handles files with inconsistent field counts, preserving timestamp fields
    """
    try:
        # First, try reading as text to analyze the structure
        with open(file_path, 'r', encoding='latin1') as f:
            lines = f.readlines()
        
        # Remove empty lines and comments
        clean_lines = [line.strip() for line in lines if line.strip() and not line.startswith('#')]
        
        if not clean_lines:
            return None
        
        # Analyze separators and field counts
        separators = ['\t', ';', ',', ' ']
        best_separator = None
        max_consistent_lines = 0
        
        for sep in separators:
            if sep in clean_lines[0]:
                # Count field consistency for this separator
                field_counts = {}
                for line in clean_lines[:100]:  # Check first 100 lines
                    count = len(line.split(sep))
                    field_counts[count] = field_counts.get(count, 0) + 1
                
                # Find most common field count
                if field_counts:
                    most_common_count = max(field_counts, key=field_counts.get)
                    consistent_lines = field_counts[most_common_count]
                    
                    if consistent_lines > max_consistent_lines:
                        max_consistent_lines = consistent_lines
                        best_separator = sep
        
        if best_separator is None:
            best_separator = '\t'  # Default fallback
        
        # Parse data manually to handle inconsistent field counts
        data = []
        headers = None
        
        for i, line in enumerate(clean_lines):
            fields = line.split(best_separator)
            
            if headers is None:
                # Use first line as headers, ensure uniqueness
                headers = []
                header_counts = {}
                for field in fields:
                    if field in header_counts:
                        header_counts[field] += 1
                        headers.append(f"{field}_{header_counts[field]}")
                    else:
                        header_counts[field] = 0
                        headers.append(field)
                continue
            
            # Skip lines that have drastically different field counts
            if len(fields) < len(headers) * 0.5:
                print(f"    Skipping line {i+1} with {len(fields)} fields (expected ~{len(headers)})")
                continue
            
            # Handle lines with extra fields more intelligently
            if len(fields) > len(headers):
                # Check if any of the header names suggest timestamp fields
                timestamp_indicators = ['timestamp', 'time', 'date', 'datum', 'zeit']
                has_timestamp_header = any(indicator in header.lower() for header in headers for indicator in timestamp_indicators)
                
                if has_timestamp_header:
                    # Keep extra fields that might be timestamp-related
                    extra_fields = len(fields) - len(headers)
                    print(f"    Line {i+1}: Found {extra_fields} extra fields, preserving potential timestamp data")
                    
                    # Extend headers with unique names for extra fields
                    for j in range(extra_fields):
                        headers.append(f"extra_field_{j+1}")
                else:
                    # Remove extra fields if no timestamp indication
                    fields = fields[:len(headers)]
            
            # Pad short lines with empty strings
            if len(fields) < len(headers):
                fields.extend([''] * (len(headers) - len(fields)))
            
            data.append(fields)
        
        if data:
            df = pd.DataFrame(data, columns=headers)
            
            # Try to convert numeric columns, but preserve timestamp columns as strings initially
            timestamp_columns = []
            for col in df.columns:
                if any(indicator in col.lower() for indicator in ['timestamp', 'time', 'date', 'datum', 'zeit']):
                    timestamp_columns.append(col)
                else:
                    try:
                        # Check if the column can be fully converted to numeric
                        numeric_series = pd.to_numeric(df[col], errors='coerce')
                        # Only convert if most values are numeric (less than 10% NaN after conversion)
                        nan_ratio = numeric_series.isna().sum() / len(numeric_series)
                        if nan_ratio < 0.1:
                            df[col] = numeric_series
                        # If too many NaN values, keep as object/string type
                    except:
                        pass
            
            # Handle timestamp columns separately
            for col in timestamp_columns:
                try:
                    # Try to parse as datetime
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                except:
                    pass
            
            return df
        
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None
    
    return None

In [8]:
def group_dat_files_by_prefix(directory_path):
    """
    Group .dat files by their first 3 digits
    """
    dat_files = glob.glob(os.path.join(directory_path, "*.dat"))
    grouped_files = defaultdict(list)
    
    for file_path in dat_files:
        filename = os.path.basename(file_path)
        
        # Extract first 3 digits from filename
        prefix = ''.join(filter(str.isdigit, filename))[:3]
        
        if len(prefix) >= 3:
            grouped_files[prefix].append(file_path)
        else:
            print(f"Warning: File {filename} doesn't start with 3 digits")
    
    return grouped_files

In [9]:
def combine_dat_files_to_csv(grouped_files, output_directory):
    """
    Combine grouped .dat files into CSV files with proper column handling
    """
    os.makedirs(output_directory, exist_ok=True)
    
    for prefix, file_list in grouped_files.items():
        print(f"Processing files with prefix {prefix}:")
        
        combined_data = []
        all_columns = set()
        
        # First pass: collect all unique column names
        dataframes = []
        for file_path in sorted(file_list):  # Sort to maintain order
            print(f"  Reading: {os.path.basename(file_path)}")
            
            df = read_dat_file(file_path)
            
            if df is not None and not df.empty:
                dataframes.append((file_path, df))
                all_columns.update(df.columns)
            else:
                print(f"    Warning: Could not read {os.path.basename(file_path)}")
        
        if dataframes:
            # Convert to sorted list for consistent column order
            all_columns = sorted(list(all_columns))
            
            print(f"  Combined columns ({len(all_columns)}): {all_columns[:5]}{'...' if len(all_columns) > 5 else ''}")
            
            # Second pass: standardize all dataframes to have the same columns
            standardized_dfs = []
            for file_path, df in dataframes:
                # Add missing columns with NaN
                for col in all_columns:
                    if col not in df.columns:
                        df[col] = pd.NA
                
                # Reorder columns to match the standard order
                df = df[all_columns]
                
                # Add source file information
                df['source_file'] = os.path.basename(file_path)
                
                standardized_dfs.append(df)
                print(f"    Standardized {os.path.basename(file_path)}: {df.shape}")
            
            # Combine all dataframes
            final_df = pd.concat(standardized_dfs, ignore_index=True)
            
            print(f"  Combined shape: {final_df.shape}")
            
            # Sort by timestamp if possible
            timestamp_cols = [col for col in final_df.columns 
                            if any(keyword in col.lower() for keyword in ['timestamp', 'time', 'date', 'datum', 'zeit'])]
            
            if timestamp_cols:
                timestamp_col = timestamp_cols[0]
                print(f"  Sorting by timestamp column: {timestamp_col}")
                try:
                    # Try to convert to datetime for sorting
                    final_df[timestamp_col] = pd.to_datetime(final_df[timestamp_col], errors='coerce')
                    final_df = final_df.sort_values(by=timestamp_col)
                    print(f"    Successfully sorted by {timestamp_col}")
                except Exception as e:
                    print(f"    Could not sort by timestamp column {timestamp_col}: {e}")
            
            # Remove duplicate rows if any
            initial_rows = len(final_df)
            final_df = final_df.drop_duplicates()
            if len(final_df) < initial_rows:
                print(f"  Removed {initial_rows - len(final_df)} duplicate rows")
            
            # Save to CSV
            output_file = os.path.join(output_directory, f"{prefix}_combined.csv")
            final_df.to_csv(output_file, index=False, sep=';', na_rep='NAN')
            
            print(f"  Saved: {output_file} ({len(final_df)} rows, {len(final_df.columns)} columns)")
            
            # Show sample of first few rows
            print(f"  Sample data preview:")
            print(f"    Columns: {list(final_df.columns)[:3]}...")
            sample_df = final_df.head(2)
            for idx, row in sample_df.iterrows():
                print(f"    Row {idx}: {row.iloc[:3].tolist()}...")
                
        else:
            print(f"  No data found for prefix {prefix}")
        
        print()

In [10]:
# Main execution
current_directory = r"C:\Users\NilsWindows\Desktop\research2.0\data\CSV AgrarMeteo\Böhringer"
dat_directory = current_directory  # Current directory contains the .dat files
output_directory = os.path.join(current_directory, "converted_csv")

print(f"Looking for .dat files in: {dat_directory}")
print(f"Output directory: {output_directory}")
print()

# Group files by prefix
grouped_files = group_dat_files_by_prefix(dat_directory)

print(f"Found {len(grouped_files)} groups of files:")
for prefix, files in grouped_files.items():
    print(f"  {prefix}: {len(files)} files")
    for file in files:
        print(f"    - {os.path.basename(file)}")
print()

# Convert to CSV
if grouped_files:
    combine_dat_files_to_csv(grouped_files, output_directory)
    print("Conversion completed!")
else:
    print("No .dat files found to process.")

Looking for .dat files in: C:\Users\NilsWindows\Desktop\research2.0\data\CSV AgrarMeteo\Böhringer
Output directory: C:\Users\NilsWindows\Desktop\research2.0\data\CSV AgrarMeteo\Böhringer\converted_csv

Found 14 groups of files:
  009: 6 files
    - 009_REN_2020.dat
    - 009_REN_2021.dat
    - 009_REN_2022.dat
    - 009_REN_2023.dat
    - 009_REN_2024.dat
    - 009_REN_2025.dat
  013: 6 files
    - 013_WAH_2020.dat
    - 013_WAH_2021.dat
    - 013_WAH_2022.dat
    - 013_WAH_2023.dat
    - 013_WAH_2024.dat
    - 013_WAH_2025.dat
  020: 6 files
    - 020_MEN_2020.dat
    - 020_MEN_2021.dat
    - 020_MEN_2022.dat
    - 020_MEN_2023.dat
    - 020_MEN_2024.dat
    - 020_MEN_2025.dat
  026: 6 files
    - 026_FRI_2020.dat
    - 026_FRI_2021.dat
    - 026_FRI_2022.dat
    - 026_FRI_2023.dat
    - 026_FRI_2024.dat
    - 026_FRI_2025.dat
  036: 6 files
    - 036_WAN_2020.dat
    - 036_WAN_2021.dat
    - 036_WAN_2022.dat
    - 036_WAN_2023.dat
    - 036_WAN_2024.dat
    - 036_WAN_2025.dat
  075: 

C:\Users\NilsWindows\AppData\Local\Temp\ipykernel_7364\2179350494.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['source_file'] = os.path.basename(file_path)
C:\Users\NilsWindows\AppData\Local\Temp\ipykernel_7364\2179350494.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['source_file'] = os.path.basename(file_path)
C:\Users\NilsWindows\AppData\Local\Temp\ipykernel_7364\2179350494.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

    Standardized 009_REN_2025.dat: (48458, 32)


C:\Users\NilsWindows\AppData\Local\Temp\ipykernel_7364\2179350494.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(standardized_dfs, ignore_index=True)


  Combined shape: (275629, 32)
  Removed 31 duplicate rows


Error: need to escape, but no escapechar set

In [ ]:
# Optional: Display summary of created CSV files
if os.path.exists(output_directory):
    csv_files = glob.glob(os.path.join(output_directory, "*.csv"))
    
    print("\nCreated CSV files:")
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, sep=';', nrows=5)  # Read first 5 rows
            print(f"\n{os.path.basename(csv_file)}:")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")
            print(f"  Sample data:")
            print(df.head(2).to_string(index=False))
        except Exception as e:
            print(f"  Error reading {os.path.basename(csv_file)}: {e}")